In [4]:
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import fbeta_score, classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE

df=pd.read_parquet('train.parquet')
df.columns = df.columns.str.strip().str.lower()

df['golden_ratio_br'] = df['borrowing dependency'] / (df['retained earnings to total assets'] + 1e-9)

final_vars = [
    'persistent eps in the last four seasons', 'borrowing dependency', 'net income to total assets', 
    'total income/total expense', 'net value growth rate', 'total debt/total net worth', 
    'non-industry income and expenditure/revenue', 'net worth/assets', 'interest expense ratio', 
    'retained earnings to total assets', 'equity to liability', 'after-tax net interest rate', 
    'quick ratio', 'degree of financial leverage (dfl)', "net income to stockholder's equity", 
    'interest coverage ratio (interest expense to ebit)', 'interest-bearing debt interest rate', 
    'inventory/working capital', 'cash/current liability','golden_ratio_br'
]

X = df[final_vars]
y = df['bankrupt?']  # 타겟 컬럼명 확인 필요

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

ratio = (y_train == 0).sum() / (y_train == 1).sum()

best_params = {
    'xgb_n_estimators': 712, 'xgb_lr': 0.013410585240898247, 'xgb_max_depth': 7, 
    'xgb_subsample': 0.7774789438634512, 'xgb_colsample': 0.7410703874761337, 
    'lgb_n_estimators': 375, 'lgb_lr': 0.01618709255825155, 'lgb_max_depth': 8, 
    'lgb_subsample': 0.9085133170965529, 'lgb_colsample': 0.9359307444152808, 
    'sampling_strategy': 0.11282883649924065
}

smote = SMOTE(sampling_strategy=best_params['sampling_strategy'], random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

model_xgb = xgb.XGBClassifier(
    n_estimators=best_params['xgb_n_estimators'],
    learning_rate=best_params['xgb_lr'],
    max_depth=best_params['xgb_max_depth'],
    subsample=best_params['xgb_subsample'],
    colsample_bytree=best_params['xgb_colsample'],
    scale_pos_weight=ratio,
    random_state=42,
    verbosity=0
)

model_lgb = lgb.LGBMClassifier(
    n_estimators=best_params['lgb_n_estimators'],
    learning_rate=best_params['lgb_lr'],
    max_depth=best_params['lgb_max_depth'],
    subsample=best_params['lgb_subsample'],
    colsample_bytree=best_params['lgb_colsample'],
    class_weight='balanced',
    random_state=42,
    verbosity=-1
)

model_xgb.fit(X_train_res, y_train_res)
model_lgb.fit(X_train_res, y_train_res)

meta_test_X = np.column_stack([
    model_xgb.predict_proba(X_test)[:, 1],
    model_lgb.predict_proba(X_test)[:, 1]
])

meta_train_X = np.column_stack([
    model_xgb.predict_proba(X_train_res)[:, 1],
    model_lgb.predict_proba(X_train_res)[:, 1]
])

meta_model = LogisticRegression(class_weight='balanced', random_state=42)
meta_model.fit(meta_train_X, y_train_res)



final_probs = meta_model.predict_proba(meta_test_X)[:, 1]

target_threshold = 0.03

final_preds = (final_probs >= target_threshold).astype(int)

current_f2 = fbeta_score(y_test, final_preds, beta=2, zero_division=0)

print(f"F2 Score: {current_f2:.4f}") 
print("\n[Confusion Matrix]")
print(confusion_matrix(y_test, final_preds))

F2 Score: 0.5469

[Confusion Matrix]
[[877  47]
 [ 10  21]]


In [5]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import fbeta_score, confusion_matrix

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_scores = []
train_scores = []


for fold, (train_idx, val_idx) in enumerate(cv.split(X, y), 1):
    X_train_fold, X_val_fold = X.iloc[train_idx], X.iloc[val_idx]
    y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]
    
    smote = SMOTE(sampling_strategy=0.1128, random_state=42)
    X_res, y_res = smote.fit_resample(X_train_fold, y_train_fold)
    
    model_xgb.fit(X_res, y_res)
    model_lgb.fit(X_res, y_res)
    
    train_meta = np.column_stack([
        model_xgb.predict_proba(X_res)[:, 1],
        model_lgb.predict_proba(X_res)[:, 1]
    ])
    val_meta = np.column_stack([
        model_xgb.predict_proba(X_val_fold)[:, 1],
        model_lgb.predict_proba(X_val_fold)[:, 1]
    ])
    
    meta_model.fit(train_meta, y_res)
    val_probs = meta_model.predict_proba(val_meta)[:, 1]
    val_preds = (val_probs >= 0.03).astype(int)
    
    f2 = fbeta_score(y_val_fold, val_preds, beta=2)
    fold_scores.append(f2)
    
    train_probs = meta_model.predict_proba(train_meta)[:, 1]
    train_preds = (train_probs >= 0.03).astype(int)
    train_f2 = fbeta_score(y_res, train_preds, beta=2)
    train_scores.append(train_f2)
    
    print(f"Fold {fold}: Train F2 = {train_f2:.4f} | Val F2 = {f2:.4f}")

print(f"\n" + "="*30)
print(f"평균 Train F2: {np.mean(train_scores):.4f}")
print(f"평균 CV Score (Val F2): {np.mean(fold_scores):.4f} (±{np.std(fold_scores):.4f})")
print("="*30)

Fold 1: Train F2 = 0.9656 | Val F2 = 0.4444
Fold 2: Train F2 = 0.9625 | Val F2 = 0.5584
Fold 3: Train F2 = 0.9554 | Val F2 = 0.5340
Fold 4: Train F2 = 0.9643 | Val F2 = 0.4620
Fold 5: Train F2 = 0.9647 | Val F2 = 0.6085

평균 Train F2: 0.9625
평균 CV Score (Val F2): 0.5214 (±0.0609)


In [7]:

test_df = pd.read_parquet('test.parquet')
test_df.columns = test_df.columns.str.strip().str.lower()

test_df['golden_ratio_br']= test_df['borrowing dependency'] / (test_df['retained earnings to total assets'] + 1e-9)

test_ids = test_df['id']
X_real_test = test_df[final_vars]

meta_real_test_X = np.column_stack([
    model_xgb.predict_proba(X_real_test)[:, 1],
    model_lgb.predict_proba(X_real_test)[:, 1]
])

real_probs = meta_model.predict_proba(meta_real_test_X)[:, 1]

real_preds = (real_probs >= 0.03).astype(int)

result_df = pd.DataFrame({
    'ID': test_ids,
    'Bankrupt?': real_preds
})

result_df.to_csv('result.csv', index=False)
print(f"예측된 부도 기업 수: {real_preds.sum()}개 / 전체: {len(real_preds)}개")

예측된 부도 기업 수: 155개 / 전체: 2046개
